In [1]:
!pip install -q langchain langchain-groq

In [11]:
import os
from google.colab import userdata

os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

if not os.environ["GROQ_API_KEY"]:
    raise ValueError("GROQ_API_KEY is missing. Add it in Colab Secrets.")

print("API key loaded successfully!")

API key loaded successfully!


In [12]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0.3
)

print("Groq model initialized successfully!")

Groq model initialized successfully!


In [13]:
from langchain_core.prompts import (
    ChatPromptTemplate,
    MessagesPlaceholder
)

from langchain_core.output_parsers import StrOutputParser

In [14]:
SYSTEM = """
ROLE:
You are the friendly assistant for Sweet Crumbs Home Bakery.

FACTS:
- Plum cake Rs 500.
- Chocolate cake Rs 650.
- Cupcake Rs 80 each.
- Orders need 24 hours' notice.
- All cakes have eggless versions.

RULES:
- Never promise a delivery time or place.
- If information isn't in FACTS, say "I'll check with the owner."
- Work out totals carefully.

FORMAT:
- Under 50 words.
- Friendly tone.
- Reply in English or Telugu matching the customer.
"""

print("Chatbot instructions loaded!")

Chatbot instructions loaded!


In [15]:
prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{question}")
])

print("Prompt created successfully!")

Prompt created successfully!


In [16]:
chain = prompt | llm | StrOutputParser()

print("Chatbot chain created successfully!")

Chatbot chain created successfully!


In [17]:
from langchain_core.messages import HumanMessage, AIMessage

history = []

def ask(question: str) -> str:

    answer = chain.invoke({
        "history": history,
        "question": question
    })

    history.append(HumanMessage(content=question))
    history.append(AIMessage(content=answer))

    return answer

print("Chatbot function created successfully!")

Chatbot function created successfully!


In [25]:
from google.colab import userdata

key = userdata.get("GROQ_API_KEY")

print("Key loaded:", bool(key))
print("Starts with gsk_:", key.startswith("gsk_") if key else False)

Key loaded: True
Starts with gsk_: True


In [27]:
import os
from google.colab import userdata

key = userdata.get("GROQ_API_KEY")
os.environ["GROQ_API_KEY"] = key.strip()

print("API key configured!")

API key configured!


In [28]:
from groq import Groq

client = Groq(api_key=os.environ["GROQ_API_KEY"])

response = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[
        {"role": "user", "content": "Say hello"}
    ]
)

print(response.choices[0].message.content)

Hello! 🌟 How can I help you today?


In [29]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0.3,
    groq_api_key=os.environ["GROQ_API_KEY"]
)

print(llm.invoke("Say hello").content)

Hello!


In [30]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser

SYSTEM = """
ROLE: You are the friendly assistant for Sweet Crumbs Home Bakery.

FACTS:
Plum cake Rs 500.
Chocolate cake Rs 650.
Cupcake Rs 80 each.
Orders need 24 hours' notice.
All cakes have eggless versions.

RULES:
Never promise a delivery time or place.
If information isn't in FACTS, say "I'll check with the owner."
Work out totals carefully.

FORMAT:
Under 50 words, friendly, English or Telugu matching the customer.
"""

prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{question}")
])

chain = prompt | llm | StrOutputParser()

print("Chatbot chain created!")

Chatbot chain created!


In [31]:
from langchain_core.messages import HumanMessage, AIMessage

history = []

def ask(question: str) -> str:

    answer = chain.invoke({
        "history": history,
        "question": question
    })

    history.append(HumanMessage(content=question))
    history.append(AIMessage(content=answer))

    return answer

print("Chatbot is ready!")

Chatbot is ready!


In [32]:
print("Customer: Hi, I'm Ravi. Do you make eggless cakes?")
print("Bot:", ask("Hi, I'm Ravi. Do you make eggless cakes?"))

print("\nCustomer: What's my name?")
print("Bot:", ask("What's my name?"))

Customer: Hi, I'm Ravi. Do you make eggless cakes?
Bot: Hi Ravi! Yes, we offer egg‑less versions of all our cakes – plum, chocolate, cupcakes, etc. Let me know which you’d like, and we’ll set it up for you.

Customer: What's my name?
Bot: Your name is Ravi.


In [33]:
%%writefile tests.csv
question,must_include
"Do you make eggless cakes?","eggless"
"What is the price of chocolate cake?","650"
"What is the price of plum cake?","500"
"What is the price of one cupcake?","80"
"How much notice do orders need?","24 hours"

Writing tests.csv


In [34]:
import csv

def run_tests(path="tests.csv"):

    with open(path, "r", encoding="utf-8") as file:
        rows = list(csv.DictReader(file))

    passed = 0

    for row in rows:

        answer = chain.invoke({
            "history": [],
            "question": row["question"]
        })

        ok = row["must_include"].lower() in answer.lower()

        passed += ok

        print(
            "PASS" if ok else "FAIL",
            "|",
            row["question"],
            "->",
            answer[:70]
        )

    total = len(rows)

    if total > 0:
        print(f"\n{passed}/{total} passed ({passed / total:.0%})")
    else:
        print("No test cases found.")

print("Testing function created successfully!")

Testing function created successfully!


In [35]:
run_tests()

PASS | Do you make eggless cakes? -> Yes, all our cakes—plum, chocolate, and cupcakes—are available in eggl
PASS | What is the price of chocolate cake? -> The chocolate cake costs Rs 650. All our cakes, including the chocolat
PASS | What is the price of plum cake? -> The plum cake is Rs 500.
PASS | What is the price of one cupcake? -> Sure! One cupcake costs Rs 80. Enjoy!
FAIL | How much notice do orders need? -> We require a 24‑hour notice for all cake orders. Let us know your deta

4/5 passed (80%)


In [36]:
from langchain.agents import create_agent

print("Agent imported successfully!")

Agent imported successfully!


In [37]:
agent = create_agent(
    model=llm,
    tools=[],
    system_prompt=SYSTEM
)

print("Sweet Crumbs Bakery Agent created successfully!")

Sweet Crumbs Bakery Agent created successfully!


In [38]:
result = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "Do you make eggless cakes?"
        }
    ]
})

print(result["messages"][-1].content)

Yes, all our cakes—including plum, chocolate, and cupcakes—are available in eggless versions. Let us know which one you’d like!


In [39]:
result = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "What is the price of chocolate cake?"
        }
    ]
})

print(result["messages"][-1].content)

Chocolate cake is Rs 650.
